# Sovereign Voice Pipeline
### Training and Deploying a Custom Piper TTS Voice on Local Hardware

**Author:** Mia L.  
**GitHub:** [miagonellm](https://github.com/miagonellm)  
**Stack:** Piper (source) · OpenAI Whisper · ONNX Runtime · Flask · ChromaDB · Ollama

**Pinned Environment:**
```
chromadb==1.5.2          faster-whisper==1.2.1       Flask==3.1.3
flask-cors==6.0.2        matplotlib==3.10.8          numpy==1.26.4
onnx==1.21.0             onnxruntime==1.18.0         onnxruntime-gpu==1.18.0
openai-whisper==20250625 pandas==1.5.3               piper-phonemize==1.1.0
piper-tts==1.4.1         pytorch-lightning==1.9.5     sentence-transformers==5.2.3
sounddevice==0.4.6       soundfile==0.13.1           torch==2.4.0+cu121
torchaudio==2.4.0+cu121  torchmetrics==1.9.0         torchvision==0.19.0+cu121
```

---

This document demonstrates how to build, train, and deploy a custom TTS voice model using Piper, a local TTS system built by Rhasspy. The pipeline operates across two environments: **local terminal** (Ubuntu 24.04) for preparation and deployment, and **Google Colab** (T5 GPU) for training only. We train on Colab because a specific pip dependency fails to build properly in the Colab environment. It's easier to isolate training there and handle everything else locally.

**Abstract**
Walk with me as we: 

Built a complete voice synthesis pipeline from raw audio to locally-deployed ONNX model. 
Wrote a silence-detection script that cuts audio at 2.0-second breath pauses for emotional range, then transcribed via OpenAI Whisper with a human validation step — listening to and confirming over an hour of original audio line by line, playing back each raw clip and correcting Whisper's transcription where it guessed wrong. 

We'll Fine-tuned a Piper TTS model on a lessac base checkpoint across 4,000 epochs at fp32 precision. Diagnosed and corrected an overfitting failure in the first training run by reducing learning rate (2e-4 → 5e-5) and epoch count (3 → 1.5). 
The Creme de la creme; export the final checkpoint to ONNX for local inference and integrated into a Flask backend with ChromaDB RAG retrieval and faster-whisper STT for real-time bidirectional speech I/O.

The result: a custom AI voice that runs entirely on local hardware, with no cloud dependency, no API costs, and no data leaving the machine.

EDIT: This Repo is currently being extended. 

---
## Phase 1 — Local Terminal Setup

Everything except training happens on your local machine. We train on Colab because a specific pip dependency causes build issues in Colab's environment. Easier to isolate training there and handle everything else locally where the environment is fully controlled.

### 1.1 Pull Piper from Source

Piper is installed **from source**, not via pip. This gives you access to the training scripts and the export tools.

In [ ]:
# ---- Local Terminal: Clone and install Piper ----

# Clone the Piper repository
git clone https://github.com/rhasspy/piper.git
cd piper/src/python

# Create and activate virtual environment (Python 3.11 REQUIRED)
python3.11 -m venv ~/piper
source ~/piper/bin/activate

# Install Piper from source
pip install -e .

# Install required dependencies
pip install pytorch-lightning==1.9.5
pip install piper-phonemize==1.1.0
pip install onnxruntime==1.18.0
pip install onnxruntime-gpu==1.18.0
pip install onnx==1.21.0

# Install espeak-ng (required for phonemization)
sudo apt-get install -y espeak-ng

# Verify
python -c "import piper_phonemize; print('piper-phonemize OK')"
echo "Piper installed from source successfully." 




### 1.2 Phonemize Gate Check

**Do this immediately after installing Piper. Before you touch any audio.**

Piper converts text → phonemes (IPA) via `espeak-ng` before training. If `piper-phonemize` is broken, has the wrong Python version, missing espeak-ng, bad build, it either fails silently and produces garbage, or produces the wrong phoneme set for your language. (making the model sound like a Sim).  Everything downstream depends on this step being correct. Catch it here or waste 20 hours training on garbage.

In [ ]:
# ---- GATE CHECK: Verify phonemization is working ----

from piper_phonemize import phonemize_espeak

# Test with a simple sentence
test = phonemize_espeak("Hello, how are you today?", "en-us")
print("Phoneme output:")
print(test)

# You should see IPA phoneme strings like:
# [['h', 'ə', 'l', 'oʊ', ...]]
#
# If you see:
#   - Empty output → espeak-ng not installed or not found
#   - ImportError → piper-phonemize not installed or Python version wrong
#   - Weird/wrong characters → wrong language code
#
# DO NOT PROCEED until this outputs clean IPA phonemes.



The Correct Phenomes will look like this. for espeak-ng. 

![description](./phen.png)

#### How to know which phoneme set your model expects

When you export a trained model to ONNX, it produces two files:
- `voice.onnx` — the model weights
- `voice.onnx.json` — the config file

The config file contains an `espeak_voice` field that tells you which phoneme set the model was trained on:

```json
{
  "espeak": {
    "voice": "en-us"
  }
}
```

**The phoneme set at inference MUST match the phoneme set used during training.** If you trained on `en-us` and try to inference with `en-gb`, the output will be garbled even though the model appears to load fine. The DNA of speech *needs* to match. 

```bash
# Check your model's expected phoneme set
cat voice.onnx.json | python3 -c "import sys,json; print(json.load(sys.stdin)['espeak']['voice'])"
```

Match this value when you test phonemization above.

### 1.3 Pull Audio/Transcription Dependencies

In [ ]:
# ---- Local Terminal: Audio + transcription tools ----

pip install faster-whisper==1.2.1
pip install openai-whisper==20250625
pip install sounddevice==0.4.6
pip install soundfile==0.13.1
pip install pydub

# For dataset analysis and graphing
pip install pandas==1.5.3
pip install numpy==1.26.4
pip install matplotlib==3.10.8
pip install seaborn

# For memory integration (Phase 3)
pip install chromadb==1.5.2
pip install sentence-transformers==5.2.3
pip install Flask==3.1.3

# SAVE YOUR WORKING ENVIRONMENT
pip freeze > requirements_working.txt
echo "Save this file. If your venv breaks, this is your recovery." 

> **Your friendly reminder:** Always run `pip freeze > requirements_working.txt` after a successful install. Environment corruption cost hours to recover from. The saved freeze file is your insurance policy.

---
### 1.4 Prepare Source Audio

Your raw audio should be:
- **30-45 minutes** of speech per voice you want to train
- Deliberately varied for emotional range (calm, warm, dry, urgent, contemplative)
- Clean recording. minimal background noise, consistent mic distance, or clear generated audio. 
- Single speaker per training set

Save your source audio as a `.wav` file.

---
### 1.5 Cut Audio into Training Clips

Two options for splitting your raw audio into individual clips:

**Option A: Hand-cut in Audacity** - manually select and export each clip. More control, more time-consuming.

**Option B: Silence-detection script** - automatically split at breath points (2.0s silence threshold). Faster, requires verification afterward.

In [ ]:
# ---- Option B: Silence-detection auto-cutter ----

from pydub import AudioSegment
from pydub.silence import split_on_silence
import os

def cut_audio_at_breaths(input_file, output_dir, min_silence_len=2000, silence_thresh=-40):
    """
    Split audio file at silence/breath points.
    
    Parameters:
    - min_silence_len: minimum silence duration (ms) to trigger a split (2000ms = 2.0s)
    - silence_thresh: silence threshold in dBFS
    """
    audio = AudioSegment.from_file(input_file)
    chunks = split_on_silence(
        audio,
        min_silence_len=min_silence_len,
        silence_thresh=silence_thresh,
        keep_silence=150  # keep 150ms of silence on each side for natural feel
    )
    
    os.makedirs(output_dir, exist_ok=True)
    
    for i, chunk in enumerate(chunks):
        chunk_path = os.path.join(output_dir, f"clip_{i:04d}.wav")
        chunk.export(chunk_path, format="wav")
        print(f"Exported: {chunk_path} ({len(chunk)/1000:.1f}s)")
    
    print(f"\nTotal clips: {len(chunks)}")
    return chunks

# Usage:
cut_audio_at_breaths("voice_raw.wav", "./voice_clips/")

After cutting, verify your clips exist and check the count:

In [ ]:
# ---- Verify clips ----


ls ./voice_clips/*.wav | wc -l        # count
ls ./voice_clips/*.wav | head -10     # preview first 10




![description](one.png)

Your terminal will look like this. #To your exact count. 

---
### 1.6 Auto-Transcribe with Whisper (Script 1: `whisper_transcribe.py`)

This script runs Whisper on every clip, writes what it heard to `metadata.csv`, and copies each clip into the dataset structure. Whisper does the heavy lifting, roughly 95% accurate on clean single-speaker audio.

In [ ]:
# ---- whisper_transcribe.py: Auto-transcription ----

import whisper
import os
import shutil

model = whisper.load_model('tiny', device='cpu')
wav_dir = os.path.expanduser('./voice_clips')
meta_path = os.path.expanduser('./voice_dataset/metadata.csv')
os.makedirs(os.path.expanduser('./voice_dataset/wav'), exist_ok=True)

with open(meta_path, 'w') as f:
    for wav_file in sorted(os.listdir(wav_dir)):
        if wav_file.endswith('.wav'):
            file_id = wav_file.replace('.wav', '')
            src = os.path.join(wav_dir, wav_file)
            dst = os.path.expanduser(f'./voice_dataset/wav/{wav_file}')
            shutil.copy2(src, dst)
            try:
                result = model.transcribe(src, fp16=False)
                text = result['text'].strip()
                if text:
                    f.write(f'{file_id}|{text}\n')
                    print(f'{file_id}: {text[:60]}...')
                else:
                    print(f'{file_id}: [SKIPPED]')
            except Exception as e:
                print(f'{file_id}: [ERROR]')

![description](trainingcuts.png)

Running this whisper_transcribe script will look like this. 

---
### 1.7 Listen and Validate (Script 2: `whisper_validate.py`)

Like I stated, Whisper is looking at 95% accuracy. But 95% means 5 out of every 100 clips have wrong transcriptions, and wrong transcriptions train wrong pronunciation. I was shooting for 100%.

So I wrote a script that plays each **original recorded clip** back through the speakers (This plays the raw audio from your .wav file), shows Whisper's guess, and lets you accept or correct it manually.

**This took over an hour of listening and confirming.** It's tedious. It's necessary. The quality of your training data is the ceiling of your model's quality.

In [ ]:
# ---- whisper_validate.py: Human validation ----
# Plays back your ORIGINAL recorded audio clips (not a model voice).
# Shows what Whisper transcribed. You accept or correct.

import sounddevice as sd
import soundfile as sf

def validate_transcriptions(clip_dir, metadata_file, output_file):
    """
    Play each original audio clip, show Whisper's transcription,
    accept (Enter) or type the correct text.
    """
    # Load existing transcriptions from whisper_transcribe.py output
    transcriptions = {}
    with open(metadata_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('|', 1)
            if len(parts) == 2:
                transcriptions[parts[0]] = parts[1]
    
    validated = []
    total = len(transcriptions)
    
    for i, (clip_id, whisper_text) in enumerate(transcriptions.items()):
        clip_path = f"{clip_dir}/{clip_id}.wav"
        
        # Play the ORIGINAL clip — your raw recorded audio
        data, samplerate = sf.read(clip_path)
        print(f"\n[{i+1}/{total}] Playing: {clip_id}")
        sd.play(data, samplerate)
        sd.wait()  # wait until clip finishes playing
        
        # Show what Whisper thinks was said
        print(f"  Whisper says: {whisper_text}")
        correction = input("  Accept (Enter) or type correction: ").strip()
        
        if correction:
            validated.append(f"{clip_id}|{correction}")
            print(f"  → Corrected.")
        else:
            validated.append(f"{clip_id}|{whisper_text}")
            print(f"  → Accepted.")
    
    # Save validated metadata
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(validated))
    
    print(f"\nValidated {total} clips. Saved to {output_file}")

# Usage:
validate_transcriptions("./voice_dataset/wav/", "./voice_dataset/metadata.csv", "./voice_dataset/metadata_validated.csv")

---
### 1.8 Verify Dataset Structure

Before uploading to Google Drive for training, verify everything is in order:

In [ ]:
# ---- Verify dataset integrity ----

import os

clip_dir = "./voice_dataset/wav/"
metadata_file = "./voice_dataset/metadata_validated.csv"

# Count clips
clips = [f for f in os.listdir(clip_dir) if f.endswith('.wav')]
print(f"Audio clips: {len(clips)}")

# Count metadata entries
with open(metadata_file, 'r') as f:
    lines = [l.strip() for l in f if l.strip()]
print(f"Metadata entries: {len(lines)}")

# Check for mismatches
clip_ids = set(os.path.splitext(f)[0] for f in clips)
meta_ids = set(l.split('|')[0] for l in lines)

missing_audio = meta_ids - clip_ids
missing_text = clip_ids - meta_ids

if missing_audio:
    print(f"\n⚠️  Metadata references clips with no audio: {missing_audio}")
if missing_text:
    print(f"\n⚠️  Audio clips with no transcription: {missing_text}")
if not missing_audio and not missing_text:
    print("\n✓ Dataset is clean. All clips have transcriptions. Ready to package.")

### 1.9 Package and Upload to Google Drive

```bash
# Create training zip (EXCLUDE lightning_logs if they exist)
cd /path/to/voice_dataset/
zip -r voice_training.zip . -x "lightning_logs/*"

# Upload to Google Drive
# Either drag-and-drop via drive.google.com or use rclone/gdrive CLI
```

> **⚠️ CRITICAL:** Never include `lightning_logs/` in your training zip. This directory contains checkpoint files that balloon the zip size to 2+ GB and cause uploads to fail silently. This is something to watch out for in your future bakes, once you've pulled your .ckpt .

---
### 1.10 Dataset Analysis

In [ ]:
# ---- Clip Duration Distribution ----

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pydub import AudioSegment

sns.set_style('darkgrid')

CLIP_DIR = "./voice_dataset/wav/"

durations = []
for f in sorted(os.listdir(CLIP_DIR)):
    if f.endswith('.wav'):
        audio = AudioSegment.from_file(os.path.join(CLIP_DIR, f))
        durations.append(len(audio) / 1000.0)

print(f"Total clips: {len(durations)}")
print(f"Total duration: {sum(durations)/60:.1f} minutes")
print(f"Mean clip length: {np.mean(durations):.1f}s")
print(f"Min: {min(durations):.1f}s | Max: {max(durations):.1f}s")

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(durations, bins=20, color='#c4a882', edgecolor='#1a1410', alpha=0.85, ax=ax)
ax.set_xlabel('Clip Duration (seconds)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Training Dataset: Clip Duration Distribution', fontsize=14, fontweight='bold')
ax.axvline(np.mean(durations), color='#8a4a2a', linestyle='--', linewidth=1.5,
           label=f'Mean: {np.mean(durations):.1f}s')
ax.legend()
plt.tight_layout()
plt.show()

---
## Phase 2 — Google Colab (Training Only)

Colab provides the GPU needed for training. **Everything in this section runs on Colab, not locally.** Expect 7-8 hours of training for 4000epochs. make sure you switch to GPU. 

### 2.1 Colab Environment Setup

In [ ]:
# ---- Colab: Session Keep-Alive ----
# Colab disconnects idle sessions. This prevents that.
# THIS IS IMPORTANT.

%javascript
function ClickConnect(){ document.querySelector('colab-connect-button').click() }
setInterval(ClickConnect, 60000)

In [ ]:
# ---- Colab: Mount Drive + Pull Data ----

from google.colab import drive
drive.mount('/content/drive')

import subprocess

# Pull training data from Drive
subprocess.run(['cp', '/content/drive/MyDrive/voice_training.zip', '/content/'])
subprocess.run(['unzip', '-o', '/content/voice_training.zip', '-d', '/content/training_data/'])

print("Training data extracted.")
!ls /content/training_data/

In [ ]:
# ---- Colab: Install Piper from Source ----
# NOT pip install piper-tts — we clone and install from source

import subprocess

subprocess.run(['git', 'clone', 'https://github.com/rhasspy/piper.git'], cwd='/content')
subprocess.run(['pip', 'install', '-e', '.', '--no-deps'], cwd='/content/piper/src/python')

!pip install pytorch-lightning==1.9.5 librosa onnxruntime onnxscript
!cd /content/piper/src/python && bash build_monotonic_align.sh
!apt-get install -y espeak-ng

# Verify
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

### 2.2 Training

> **⚠️ CRITICAL: `--default_root_dir` MUST point to Google Drive.**  
> If Colab disconnects and your checkpoints were saved to Colab's local storage, your entire training run is **gone**. Unrecoverable. Saving to Drive means you can resume from the last checkpoint.
>
> **⚠️ CRITICAL: `--checkpoint-epochs 100`** saves a checkpoint every 100 epochs. This is your safety net. Do not skip this flag.

> Dont learn the hard way, make sure you write these in. 

In [ ]:
# ---- Colab: Launch Training ----

!cd /content/piper/src/python && python3 -m piper_train \
  --dataset-dir /content/training_data/ \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.0 \
  --num-test-examples 0 \
  --max_epochs 4000 \
  --resume_from_checkpoint /content/drive/MyDrive/voice_final/lessac_base.ckpt \
  --checkpoint-epochs 100 \
  --precision 32 \
  --default_root_dir /content/drive/MyDrive/voice_final/


![description](collab.png)
The picture diplayed will show you what your monitor will look like after training has completed. 


In [ ]:

unzip lightning_logs.zip -d ./lightning_logs/

### 2.3 Training Monitoring & Loss Analysis

The training loss measures how far the model's output is from the target audio. A healthy run shows rapid early drop, gradual convergence, then stabilization.

#### Checkpoint Comparison (Empirical - by ear)

Loss data was logged to Colab's console output during training and **was not saved** to a persistent file. 

We still have resolution. The evaluation method was empirical. exporting multiple checkpoints and listening to each one to find the sweet spot:

| Checkpoint | Epoch | Result |
|---|---|---|*
| lessac base | 2164 | Perfect English, no custom voice |
| Early fine-tune | ~2899 | Sounds like lessac base model with your skin. , English intact |
| Mid fine-tune | ~3099 | Sounds like a mixture of your base model and lessac's base. drifting from both, identifying for neither; something different. |
| Late fine-tune | ~3499 | Beginning to sound like your model now, lessac base undetectable |
| Full fine-tune | ~3999 -5999 | Full custom voice, can achieve highly speech consistency and clarity with higher epochs and, different data sets  |   ← **best**

The sweet spot was in 4000-5999 range per-training data compilation. Beyond that, you can begin to hear the models deprecation.
However, you can train newly structured dataset's on top of Piper. This means if you take the same voice achitecture and create a new performance (a new data set), You can take you final ckpt (4000-5999 or which ever epoch you choose) you can train again, without the risk of oveer training.  

#### Loss Curve Script (template for future bakes)

The graph script below is a template. for this repo, **On the next training committ**.

To see this graphical return, loss will be logged to `metrics.csv` via PyTorch Lightning callbacks and plotted here with real data. Screenshots of the completed graphs will accompany this section.

In [ ]:
# ---- Training Loss: Bake 1 vs Bake 2 ----
# REPLACE with your real values from lightning_logs/metrics.csv
# To load real data:
# import pandas as pd
# df = pd.read_csv('/content/drive/MyDrive/voice_final/lightning_logs/version_0/metrics.csv')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')

# Placeholder data — REPLACE WITH REAL VALUES
np.random.seed(42)
epochs_1 = np.arange(0, 4000, 50)
loss_1 = 2.5 * np.exp(-epochs_1/400) + 0.15 + np.random.normal(0, 0.03, len(epochs_1))
loss_1[40:] = loss_1[40:] - np.linspace(0, 0.12, len(loss_1[40:]))

epochs_2 = np.arange(0, 4000, 50)
loss_2 = 2.5 * np.exp(-epochs_2/500) + 0.22 + np.random.normal(0, 0.02, len(epochs_2))
loss_2[30:] = loss_2[30:] + np.random.normal(0, 0.01, len(loss_2[30:]))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(epochs_1, loss_1, color='#8a4a2a', linewidth=1.5, alpha=0.9,
        label='Bake 1 (lr=2e-4, 3 epochs) — overfit')
ax.plot(epochs_2, loss_2, color='#4a6b3a', linewidth=1.5, alpha=0.9,
        label='Bake 2 (lr=5e-5, 1.5 epochs) — stable')
ax.axhline(y=0.22, color='#c4a882', linestyle=':', linewidth=1, alpha=0.7,
           label='Convergence threshold')
ax.axvline(x=2900, color='#5a3d6e', linestyle='--', linewidth=1, alpha=0.6,
           label='Best checkpoint (epoch 2899)')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Training Loss', fontsize=12)
ax.set_title('Training Loss: Bake 1 (Overfit) vs Bake 2 (Corrected)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("""
Diagnosis:
- Bake 1: Learning rate (2e-4) too aggressive. Loss continued decreasing
  past convergence, producing repetitive/looping output at inference.
- Bake 2: Reduced lr to 5e-5 and limited to 1.5 epochs. Loss stabilized.
  Output maintained variety and natural prosody.
- Best checkpoint selected at epoch 2899 based on loss plateau +
  qualitative listening tests.
""")

### 2.4 How Overfitting Manifested

Overfitting in TTS doesn't look like overfitting in classification. There's no accuracy metric to watch.

| Signal | Healthy | Overfit |
|---|---|---|
| Output variety | Different prosody per sentence | Same rhythm regardless of input |
| Long sentences | Natural pacing | Loops or repeats phrases |
| Unseen words | Reasonable attempt | Garbled or monotone |
| Loss curve | Flattens and stabilizes | Keeps decreasing past plateau |

The diagnosis came from **listening**, not from metrics alone. Loss curves confirmed what the ear detected first.

---
## Phase 3 — Back to Local (Export & Deploy)

Training is done. Your time in collabs kitchen has finished. The checkpoint lives on Google Drive. Now we bring it home.

### 3.1 Pull Checkpoint from Google Drive

Download the `.ckpt` file you want to export from Google Drive. Drag-and-drop from drive.google.com to your local machine. Move it to your working directory:

```bash
mv ~/Downloads/epoch=XXXX-step=XXXXXXX.ckpt ~/VoiceTraining/
```

### 3.2 Patch Export Script (one time only)

Newer versions of PyTorch block loading checkpoints by default. Patch Piper's export script to allow it:

```bash
sed -i '1s/^/import pathlib\nimport torch.serialization\ntorch.serialization.add_safe_globals([pathlib.PosixPath])\n/' ~/VoiceTraining/piper/src/python/piper_train/export_onnx.py
```

You only need to do this once. After patching, the export script works permanently.

### 3.3 Export Checkpoint to ONNX

The training output is a `.ckpt` file (checkpoint). Piper inference requires `.onnx`. The export happens **locally**, not on Colab, the torch version mismatches between Colab and local cause export failures.

```bash
# Activate local piper venv
cd ~/VoiceTraining/piper/src/python
source ~/piper/bin/activate

# Export checkpoint to ONNX
python3 -m piper_train.export_onnx \
    ~/VoiceTraining/epoch=XXXX-step=XXXXXXX.ckpt \
    ~/VoiceTraining/your_voice.onnx

# The export produces ONLY the .onnx file.
# The .json config must be copied manually from your training data:
cp ./training_data/config.json ~/VoiceTraining/your_voice.onnx.json
```

> **Note:** The `.onnx.json` is not auto-generated by the export. It's the config file from your training dataset folder. Copy it and rename it to match your `.onnx` filename. They must sit in the same directory.

### 3.4 Verify Phoneme Set After Export

Check that your exported model expects the same phoneme set you trained on:

```bash
cat ~/VoiceTraining/your_voice.onnx.json | python3 -c \"import sys,json; print(json.load(sys.stdin)['espeak']['voice'])\"
```

Should output `en-us`. If it doesn't match what you tested in the phonemize gate check (Section 1.2), the voice will sound garbled at inference.

### 3.5 Test the Voice

```bash
echo \"You've completed your task, Great. Lets do the next one, soon.\" | python3 -m piper --model ~/VoiceTraining/your_voice.onnx --output_file ~/test.wav && ffplay -autoexit ~/test.wav
```

If it speaks, it worked. If it errors on export, check your torch version. The `piper` venv must have the same major torch version used during training.

### 3.2 Integration into Voice Pipeline

The trained ONNX model connects to the full system through a single Flask process. The minimum imports for voice to work with an LLM:

In [ ]:
# ---- Minimum imports for voice-enabled LLM ----

from piper.voice import PiperVoice       # speak (TTS)
from faster_whisper import WhisperModel   # hear (STT)
import requests                           # talk to Ollama (LLM)
import wave                               # audio format
import sounddevice as sd                  # play/record audio
import soundfile as sf                    # read/write audio files
import chromadb                           # remember (vector memory)
from sentence_transformers import SentenceTransformer  # embed text

# 8 imports. Hear → Think → Remember → Speak.

from piper.voice import PiperVoice       # speak (TTS)
from faster_whisper import WhisperModel   # hear (STT)
import requests                           # talk to Ollama (LLM)
import wave                               # audio format
import sounddevice as sd                  # play/record audio
import soundfile as sf                    # read/write audio files
import chromadb                           # remember (vector memory)
from sentence_transformers import SentenceTransformer  # embed text

def ask_model(prompt, model="your_model"):
    response = requests.post("http://localhost:11434/api/generate", json={
        "model": model,
        "prompt": prompt,
        "stream": False
    })
    return response.json()["response"]

    #This is what your pipelines base will look like.

### 3.3 Pipeline Architecture

```
┌─────────────────────────────────────────────────────────┐
│                  SOVEREIGN VOICE PIPELINE               │
│                                                         │
│   ┌──────────┐    ┌──────────┐    ┌───────────────┐     │
│   │ Mic /    │───▶│ faster-  │───▶│ Ollama        │     │
│   │ Audio In │    │ whisper  │    │ (Llama 3.2)   │     │
│   └──────────┘    │ STT      │    │               │     │
│                   └──────────┘    │  ┌───────────┐│     │
│                                   │  │ ChromaDB  ││     │
│                                   │  │ RAG       ││     │
│   ┌──────────┐    ┌──────────┐    │  │ retrieval ││     │
│   │ Speaker /│◀───│ Piper    │◀── │  └───────────┘│     │
│   │ Audio Out│    │ TTS      │    └───────────────┘     │
│   └──────────┘    │ (ONNX)   │                          │
│                   └──────────┘                          │
│                                                         │
│   Flask ◀──────────────────────────────────────────▶    │
│   (talk_piper.py)         All routes, one process       │
└─────────────────────────────────────────────────────────┘
```

All components run through a single Flask application. Terminal and GUI share the same model, memory, and voice pipeline. No separate processes, no inter-service communication, no sync issues.

### 3.4 Inference Performance

In [ ]:
# ---- Inference Latency Test ----
#Check the stats of your LLM's response time.
# REPLACE with actual measurements from your hardware

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')

# REPLACE with real measurements
sentence_lengths = [10, 20, 40, 60, 80, 100, 150, 200, 300]
latencies_ms = [45, 62, 88, 110, 135, 158, 210, 275, 390]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sentence_lengths, latencies_ms, 'o-', color='#5a3d6e', linewidth=2, markersize=8)
ax.fill_between(sentence_lengths, latencies_ms, alpha=0.1, color='#5a3d6e')
ax.axhline(y=200, color='#8a4a2a', linestyle='--', linewidth=1, label='200ms real-time threshold')
ax.set_xlabel('Input Length (characters)', fontsize=12)
ax.set_ylabel('Synthesis Latency (ms)', fontsize=12)
ax.set_title('Inference Latency: ONNX Model on Local GPU', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("Sub-200ms inference for typical utterances (< 100 characters).")
print("Suitable for real-time conversational use.")

---
## 4. Results & Demo

### 4.1 Plug In and Listen

Once exported, test your voice directly from the terminal:

```bash
echo \"You've completed your task, Great. Lets do the next one, soon.\" | python3 -m piper --model ~/VoiceTraining/your_voice.onnx --output_file ~/test.wav && ffplay -autoexit ~/test.wav
```

### 4.2 Audio Samples

| Sample | Description |
|---|---|
| [Base lessac](./samples/base_sample.wav) | Default pre-trained voice — generic, flat |
| [Trained voice](./samples/trained_sample.wav) | Fine-tuned model — distinct character |

### 4.3 Demo Video

▶️ [Watch: Piper TTS model running on Finetuned 3bil Ollama. (SHORT EXCHANGE)](https://youtu.be/46iRkHK_F34?si=hTnF9v2n_sVWO9_W)

▶️ [Watch: Piper TTS model running on Finetuned 3bil Ollama. (LONG EXCHANGE)](https://youtu.be/uyaW-_78wIc?si=wo4M1W6Rc_BdCOQ0)

Check out a Demo video previewing Dual identity interaction. A *learning* 3bil Ollama model, RAG and fine tune scafffolding supports the Identity. 

The video demonstrates real-time voice interaction running entirely from the terminal on local hardware. Whisper transcription → 3b Ollama model response → Piper TTS output. Two distinct trained voices speaking from the same pipeline. No internet required.

### 4.4 Why Sovereign/Offline Matters

- **No API costs.** Inference is free after training.
- **No rate limits.** Speak as much as you want.
- **No data exfiltration.** Your conversations never leave your machine.
- **No vendor dependency.** The model can't be deprecated, revoked, or updated out from under you.
- **Persistent.** The voice is yours permanently.

---
## 5. Lessons Learned

| Problem | Cause | Fix |
|---|---|---|
| Bake 1 produced repetitive output | lr too high (2e-4), too many epochs (3) | Reduced to lr=5e-5, 1.5 epochs |
| Lost a 20+ hour training run | Colab disconnected, checkpoints on local storage | Always `--default_root_dir` to Google Drive |
| ONNX export failed on Colab | torch version mismatch between Colab and local | Export locally in controlled `piper` venv |
| `piper-phonemize` import error | Python 3.12 breaks the C extension | Pin to Python 3.11 |
| Training zip upload failed silently | `lightning_logs/` directory inflated zip to 2+ GB | Always exclude `lightning_logs/` from zips |
| `piper` venv corrupted after system update | System Python upgrade overwrote venv links | Keep a saved `pip freeze` for recovery |
| Whisper transcription ~95% accurate | Whisper guesses wrong on ~5% of clips | Human validation. listened to every single clip |
| Phoneme set mismatch at inference | Trained on en-us, tested with en-gb | Check `voice.onnx.json` espeak voice field must match |

---

*Built with patience, failure, and pesistence. Give your model the ability to communicate with you auditorially; reach new adaptive forms of communication. 

**Contact:** [GitHub](https://github.com/miagonellm) · [Portfolio](https://miagonellm.github.io/222datascience.github.io/)
